In [104]:
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
import json
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain_core.runnables import  RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [105]:
load_dotenv()

True

In [106]:
def load_data(json_path="recipes.json"):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    docs = [Document(page_content=entry['content'], metadata={'title': entry['title'],'type':entry['type']}) for entry in data]
    return docs

In [107]:
documents = load_data()
print(documents)

[Document(metadata={'title': 'Dal Bhat', 'type': 'main course'}, page_content='Ingredients: 1 cup rice, 1/2 cup lentils (dal), 1/2 tsp turmeric, salt to taste, vegetables (spinach, cauliflower, etc.), tomato achar. Instructions: 1. Rinse lentils and boil with turmeric and salt until soft. 2. In another pot, cook rice until fluffy. 3. Stir-fry seasonal vegetables with spices. 4. Serve rice with dal, vegetables, and achar.'), Document(metadata={'title': 'Momo (Dumplings)', 'type': 'main course'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, ginger, onion, soy sauce, salt to taste. Instructions: 1. Knead dough using flour and water. 2. Mix filling ingredients together. 3. Roll dough, fill with mixture, and shape into dumplings. 4. Steam for 10 to 15 minutes or deep-fry. 5. Serve with spicy tomato achar.'), Document(metadata={'title': 'Sel Roti', 'type': 'snack'}, page_content='Ingredients: 2 cups rice flour, 1 ripe banana, 1/4 cup sugar, 1/2 cup wate

In [108]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(documents)

In [109]:
print(chunks)

[Document(metadata={'title': 'Dal Bhat', 'type': 'main course'}, page_content='Ingredients: 1 cup rice, 1/2 cup lentils (dal), 1/2 tsp turmeric, salt to taste, vegetables (spinach, cauliflower, etc.), tomato achar. Instructions: 1. Rinse lentils and boil with turmeric and salt until soft. 2. In another pot, cook rice until fluffy. 3. Stir-fry seasonal vegetables with spices. 4. Serve rice with dal, vegetables, and achar.'), Document(metadata={'title': 'Momo (Dumplings)', 'type': 'main course'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, ginger, onion, soy sauce, salt to taste. Instructions: 1. Knead dough using flour and water. 2. Mix filling ingredients together. 3. Roll dough, fill with mixture, and shape into dumplings. 4. Steam for 10 to 15 minutes or deep-fry. 5. Serve with spicy tomato achar.'), Document(metadata={'title': 'Sel Roti', 'type': 'snack'}, page_content='Ingredients: 2 cups rice flour, 1 ripe banana, 1/4 cup sugar, 1/2 cup wate

In [110]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


In [111]:
from langchain.vectorstores import Chroma
# vectorstore = Chroma.from_documents(chunks,embedding_model)
vectorstore = Chroma.from_documents(chunks,embedding_model)

In [112]:
metadata_field_info =[
      AttributeInfo(name="title", description="The title of the recipe", type="string"),
      AttributeInfo(name="type", description="The type of the recipe", type="string"),
]

document_content_description = "Brief summary about food recipe"

## Hugging face LLM model

In [113]:
model = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation"
)

llm = ChatHuggingFace(llm=model,temperature=0)


## Gemini api

In [114]:
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash',api_key=os.getenv('GEMINI_API_KEY'),temperature=0.7)

In [115]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info
)

In [116]:
retriever.invoke("How do I make Momo (Dumplings)?")

[Document(metadata={'type': 'main course', 'title': 'Momo (Dumplings)'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, ginger, onion, soy sauce, salt to taste. Instructions: 1. Knead dough using flour and water. 2. Mix filling ingredients together. 3. Roll dough, fill with mixture, and shape into dumplings. 4. Steam for 10 to 15 minutes or deep-fry. 5. Serve with spicy tomato achar.'),
 Document(metadata={'type': 'main course', 'title': 'Momo'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, ginger, onion, soy sauce, salt to taste. Instructions: 1. Knead dough using flour and water. 2. Mix filling ingredients together. 3. Roll dough, fill with mixture, and shape into dumplings. 4. Steam for 10 to 15 minutes or deep-fry. 5. Serve with spicy tomato achar.'),
 Document(metadata={'type': 'main course', 'title': 'Momo (Dumplings)'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, g

In [117]:
prompt_template = """
You are an expert cooking assistant.

Use the following information extracted from Nepali food recipes to answer the question.

Context:
{context}

Question:
{question}

Instructions:
- Answer based only on the provided context.
- If the answer cannot be found in the context, respond: "Sorry, I don't have that information."
- Be clear and concise.
- Provide step-by-step instructions if the question asks for a recipe.

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [122]:
parser = StrOutputParser()

# Final chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser
)

query = "How do I make Momo (Dumplings)?"
response = chain.invoke(query)

print("Answer:", response)


Answer: To make Momo (Dumplings), follow these steps:

1. Knead dough using flour and water.
2. Mix filling ingredients together, which include minced chicken or vegetables, garlic, ginger, onion, soy sauce, and salt to taste.
3. Roll the dough, fill it with the mixture, and shape into dumplings.
4. Steam the dumplings for 10 to 15 minutes or deep-fry them.
5. Serve the Momo with spicy tomato achar.
